In [2]:
import pandas as pd
import logging 
import os
import numpy as np

import src.constants as C
from src.preprocessing import process_store_data, drop_closed
from src.features import attach_store_data, make_features, make_targets
from src.engine import nested_cv

logging.basicConfig(
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(C.LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,  # ensure no duplicate handlers if this cell is re-run in a notebookcls
)

logger = logging.getLogger(__name__)

os.makedirs(C.LOG_DIR, exist_ok=True)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

c:\Users\m_kal\anaconda3\envs\rossmann\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
stores = pd.read_csv(C.STORE_FILE)
stores = process_store_data(stores)

df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)

df['Sales'] = df['Sales'].apply(np.log1p)
df = attach_store_data(df, stores).sort_values(['Store', 'Date'])
X = make_features(df, C.LAGS, C.ROLL_WINDOWS, C.DIFFS)
y = make_targets(df[['Date', 'Store', 'Sales']], C.FORECAST_HORIZON)
X, y = drop_closed(X, y)

logger.info(f"Transformed dataset: {len(X)} samples, {X.shape[1]} features")

C:\Users\m_kal\AppData\Local\Temp\ipykernel_16368\752888049.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
INFO:__main__:Transformed dataset: 844392 samples, 77 features


In [4]:
pd.concat([
    X.dtypes,
    X.isna().sum()/len(X),
    X.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('./artifacts/feature_summary.csv')

In [6]:
#==================================================================================
num_stores = 3
X_small = X.xs(slice(None, num_stores), level="Store", drop_level=False)
y_small = y.xs(slice(None, num_stores), level="Store", drop_level=False)
#==================================================================================

nested_cv(X_small, y_small)


XGBoostError: [12:36:17] C:\actions-runner\_work\xgboost\xgboost\src\data\data.cc:1173: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`

In [ ]:
X_small